In [2]:
import pandas as pd
import requests
import time
import json

In [22]:
BASE_URL = "https://fantasy.premierleague.com/api/"

def get_bootstrap_data():
    """Get the master list of players, teams, and positions."""
    r = requests.get(BASE_URL + "bootstrap-static/")
    r.raise_for_status()
    return r.json()
 
 
def get_player_history(player_id):
    """Get one player's full gameweek-by-gameweek history."""
    r = requests.get(BASE_URL + f"element-summary/{player_id}/")
    r.raise_for_status()
    return r.json()["history"]

def get_gameweek_data(gameweek):
    """Get all players' data for a specific gameweek."""
    r = requests.get(BASE_URL + f"event/{gameweek}/live/")
    r.raise_for_status()
    return r.json()["elements"]

In [28]:
response_events = get_bootstrap_data()
print(response_events.keys())

dict_keys(['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements'])


In [32]:
get_player_history(1)

[{'element': 1,
  'fixture': 1,
  'opponent_team': 7,
  'total_points': 6,
  'was_home': True,
  'kickoff_time': '2026-08-21T19:00:00Z',
  'team_h_score': 3,
  'team_a_score': 0,
  'round': 1,
  'modified': False,
  'minutes': 90,
  'goals_scored': 0,
  'assists': 0,
  'clean_sheets': 1,
  'goals_conceded': 0,
  'own_goals': 0,
  'penalties_saved': 0,
  'penalties_missed': 0,
  'yellow_cards': 0,
  'red_cards': 0,
  'saves': 1,
  'bonus': 0,
  'bps': 24,
  'influence': '11.8',
  'creativity': '0.0',
  'threat': '0.0',
  'ict_index': '1.2',
  'clearances_blocks_interceptions': 1,
  'recoveries': 8,
  'tackles': 0,
  'defensive_contribution': 0,
  'starts': 1,
  'expected_goals': '0.00',
  'expected_assists': '0.00',
  'expected_goal_involvements': '0.00',
  'expected_goals_conceded': '0.20',
  'value': 60,
  'transfers_balance': 0,
  'selected': 3347153,
  'transfers_in': 0,
  'transfers_out': 0},
 {'element': 1,
  'fixture': 20,
  'opponent_team': 2,
  'total_points': 0,
  'was_home': 

In [23]:
response = get_gameweek_data(1)
print(response[0].keys())

dict_keys(['id', 'stats', 'explain', 'modified'])


In [25]:
response[410]

{'id': 411,
 'stats': {'minutes': 90,
  'goals_scored': 0,
  'assists': 0,
  'clean_sheets': 0,
  'goals_conceded': 1,
  'own_goals': 0,
  'penalties_saved': 0,
  'penalties_missed': 0,
  'yellow_cards': 0,
  'red_cards': 0,
  'saves': 0,
  'bonus': 0,
  'bps': 6,
  'influence': '3.8',
  'creativity': '5.2',
  'threat': '50.0',
  'ict_index': '5.9',
  'clearances_blocks_interceptions': 0,
  'recoveries': 3,
  'tackles': 0,
  'defensive_contribution': 3,
  'starts': 1,
  'expected_goals': '0.74',
  'expected_assists': '0.02',
  'expected_goal_involvements': '0.76',
  'expected_goals_conceded': '0.65',
  'total_points': 2,
  'in_dreamteam': False,
  'played': True},
 'explain': [{'fixture': 8,
   'stats': [{'identifier': 'minutes',
     'points': 2,
     'value': 90,
     'points_modification': 0}]}],
 'modified': False}

In [15]:
for r in response['elements']:
    if r['cost_change_start_fall'] != 0:
        print(r['web_name'])

Calafiori
Hincapie
Eze
Ødegaard
Madueke
Merino
Martinelli
Zubimendi
Dowman
Gyökeres
Bruno G.
Watkins
Kayode
M.Sangaré
F.Kadıoğlu
De Cuyper
Minteh
Palmer
João Pedro
Chadi Riad
Sarr
M.França
Aït-Nouri
Cherki
Doku
Anderson
Dorgu
Amad
Zirkzee
Gibbs-White
Hudson-Odoi
Maddison
P.M.Sarr
Adingra


In [53]:
response["element_types"]

[{'id': 1,
  'plural_name': 'Goalkeepers',
  'plural_name_short': 'GKP',
  'singular_name': 'Goalkeeper',
  'singular_name_short': 'GKP',
  'squad_select': 2,
  'squad_min_select': None,
  'squad_max_select': None,
  'squad_min_play': 1,
  'squad_max_play': 1,
  'ui_shirt_specific': True,
  'sub_positions_locked': [12],
  'element_count': 67},
 {'id': 2,
  'plural_name': 'Defenders',
  'plural_name_short': 'DEF',
  'singular_name': 'Defender',
  'singular_name_short': 'DEF',
  'squad_select': 5,
  'squad_min_select': None,
  'squad_max_select': None,
  'squad_min_play': 3,
  'squad_max_play': 5,
  'ui_shirt_specific': False,
  'sub_positions_locked': [],
  'element_count': 205},
 {'id': 3,
  'plural_name': 'Midfielders',
  'plural_name_short': 'MID',
  'singular_name': 'Midfielder',
  'singular_name_short': 'MID',
  'squad_select': 5,
  'squad_min_select': None,
  'squad_max_select': None,
  'squad_min_play': 2,
  'squad_max_play': 5,
  'ui_shirt_specific': False,
  'sub_positions_lock

In [33]:
import duckdb
from pathlib import Path
import os

PROJECT_ROOT = os.getcwd()
PLAYERS_PATH = PROJECT_ROOT + "/data/processed/players.parquet"
GW_GLOB = PROJECT_ROOT + "/data/processed/gw_*.parquet"
DB_PATH = PROJECT_ROOT + "/fpl.duckdb"

def find_player(name):
    pattern = f"%{name}%"
    con = duckdb.connect(str(PROJECT_ROOT + "/fpl.duckdb"))
    result = con.execute(f"""
        SELECT player_id, first_name, last_name, name, position
        FROM read_parquet('{PLAYERS_PATH}')
        WHERE name ILIKE ? OR first_name ILIKE ? OR last_name ILIKE ?
    """, [pattern, pattern, pattern]).df()
    con.close()
    return result

def get_player_gameweek(name, gameweek):
    pattern = f"%{name}%"
    con = duckdb.connect(str(DB_PATH))
    result = con.execute(f"""
        SELECT p.name, p.position, g.*
        FROM read_parquet('{GW_GLOB}') g
        JOIN read_parquet('{PLAYERS_PATH}') p ON g.player_id = p.player_id
        WHERE (p.name ILIKE ? OR p.first_name ILIKE ? OR p.last_name ILIKE ?)
          AND g.round = ?
    """, [pattern, pattern, pattern, gameweek]).df()
    con.close()
    return result

In [35]:
find_player("Tarkowski")

get_player_gameweek("Tarkowski", 1)

,name,position,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,...,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out,player_id,team
0,Tarkowski,DEF,3,8,6,True,2026-08-22T14:00:00Z,2,0,1,...,0.00,0.05,1.96,60,0,765229,0,0,229,Everton
